# BrainVision AI — EfficientNetB0 Training Pipeline

**Project:** BrainVision AI  
**Task:** Four-class brain MRI classification  
**Classes:** Glioma, Meningioma, Pituitary, No Tumor  
**Architecture:** EfficientNetB0 transfer learning  
**Framework:** TensorFlow / Keras

This notebook provides the complete training, fine-tuning, evaluation, and export pipeline for the BrainVision AI project.

> **Educational use only:** This model is not a medical device and must not be used for clinical diagnosis, treatment, or patient-care decisions.

## 1. Notebook Workflow

1. Configure the environment and deterministic seeds.
2. Locate and validate the Masoud Nickparvar dataset.
3. Inspect class distributions and image integrity.
4. Build efficient TensorFlow input pipelines.
5. Train an ImageNet-pretrained EfficientNetB0 classifier.
6. Fine-tune the upper backbone layers.
7. Evaluate the final model on the held-out test directory.
8. Generate confusion-matrix, ROC, precision-recall, and error-analysis artifacts.
9. Save the trained model, labels, metrics, history, and run metadata.

In [ ]:
from __future__ import annotations

import gc
import json
import os
import platform
import random
import sys
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import yaml
from PIL import Image, UnidentifiedImageError
from sklearn.metrics import (
    auc,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_curve,
)
from sklearn.preprocessing import label_binarize
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

print("Python:", sys.version.split()[0])
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("Platform:", platform.platform())
print("GPU devices:", tf.config.list_physical_devices("GPU"))

## 2. Paths and Configuration

The notebook is designed to run from either:

- the repository root,
- the `training/` directory,
- Google Colab, or
- a local Jupyter environment.

Set `DATASET_ROOT` manually only when automatic discovery does not find the dataset.

In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / "config" / "training_config.yaml").exists():
            return candidate
    # Fallback for a notebook opened directly from the training directory.
    if current.name == "training":
        return current.parent
    return current

PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "config" / "training_config.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Configuration file not found: {CONFIG_PATH}\n"
        "Run this notebook from the BrainVisionAI repository."
    )

with CONFIG_PATH.open("r", encoding="utf-8") as file:
    CONFIG = yaml.safe_load(file)

PROJECT_ROOT

In [ ]:
# Optional manual override. Leave as None for automatic discovery.
DATASET_ROOT: Path | None = None

candidate_roots = [
    PROJECT_ROOT / CONFIG["dataset"]["expected_root"],
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "dataset",
    Path("/content/brain-tumor-mri-dataset"),
    Path("/content/Brain Tumor MRI Dataset"),
    Path("/content/drive/MyDrive/brain-tumor-mri-dataset"),
    Path("/kaggle/input/brain-tumor-mri-dataset"),
]

def has_expected_structure(path: Path) -> bool:
    return (path / CONFIG["dataset"]["training_dir"]).is_dir() and (
        path / CONFIG["dataset"]["testing_dir"]
    ).is_dir()

if DATASET_ROOT is None:
    for candidate in candidate_roots:
        if has_expected_structure(candidate):
            DATASET_ROOT = candidate.resolve()
            break
        # Some users point to a parent folder containing the actual dataset folder.
        if candidate.is_dir():
            for child in candidate.iterdir():
                if child.is_dir() and has_expected_structure(child):
                    DATASET_ROOT = child.resolve()
                    break
        if DATASET_ROOT is not None:
            break

if DATASET_ROOT is None:
    searched = "\n".join(f"  - {path}" for path in candidate_roots)
    raise FileNotFoundError(
        "Could not locate the Brain Tumor MRI Dataset.\n"
        "Expected Training/ and Testing/ subdirectories.\n"
        f"Searched:\n{searched}\n\n"
        "Set DATASET_ROOT manually in this cell and run it again."
    )

TRAIN_DIR = DATASET_ROOT / CONFIG["dataset"]["training_dir"]
TEST_DIR = DATASET_ROOT / CONFIG["dataset"]["testing_dir"]
MODEL_DIR = PROJECT_ROOT / "model"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
LOG_DIR = PROJECT_ROOT / "logs"

for directory in (MODEL_DIR, ARTIFACT_DIR, LOG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset root:", DATASET_ROOT)
print("Training directory:", TRAIN_DIR)
print("Testing directory:", TEST_DIR)

## 3. Reproducibility and Runtime Configuration

Mixed precision is enabled only when a compatible GPU is available. CPU training remains supported.

In [ ]:
SEED = int(CONFIG["project"]["seed"])
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception as exc:
    print("Deterministic operation mode could not be enabled:", exc)

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

if gpus:
    keras.mixed_precision.set_global_policy("mixed_float16")
else:
    keras.mixed_precision.set_global_policy("float32")

print("Random seed:", SEED)
print("Mixed precision policy:", keras.mixed_precision.global_policy())

## 4. Dataset Validation

This section verifies:

- required class folders,
- supported image extensions,
- readable image files,
- RGB conversion compatibility,
- image dimensions,
- and class counts.

Corrupt files are reported before training starts.

In [ ]:
EXPECTED_CLASSES = list(CONFIG["dataset"]["classes"])
DISPLAY_NAMES = dict(CONFIG["dataset"]["display_names"])
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}

def normalize_class_name(name: str) -> str:
    return name.lower().replace(" ", "").replace("_", "").replace("-", "")

def resolve_class_directories(base_dir: Path) -> dict[str, Path]:
    available = {
        normalize_class_name(path.name): path
        for path in base_dir.iterdir()
        if path.is_dir()
    }
    resolved: dict[str, Path] = {}
    aliases = {
        "glioma": {"glioma", "gliomatumor"},
        "meningioma": {"meningioma", "meningiomatumor"},
        "notumor": {"notumor", "no_tumor", "normal"},
        "pituitary": {"pituitary", "pituitarytumor"},
    }
    for expected in EXPECTED_CLASSES:
        matches = [
            path for normalized, path in available.items()
            if normalized in {normalize_class_name(alias) for alias in aliases[expected]}
        ]
        if len(matches) != 1:
            raise ValueError(
                f"Expected exactly one directory for class '{expected}' under {base_dir}, "
                f"but found: {[path.name for path in matches]}"
            )
        resolved[expected] = matches[0]
    return resolved

train_class_dirs = resolve_class_directories(TRAIN_DIR)
test_class_dirs = resolve_class_directories(TEST_DIR)

print("Resolved training classes:")
for label, directory in train_class_dirs.items():
    print(f"  {label:12s} -> {directory.name}")

print("\nResolved testing classes:")
for label, directory in test_class_dirs.items():
    print(f"  {label:12s} -> {directory.name}")

In [ ]:
def scan_dataset(class_dirs: dict[str, Path], split_name: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    records: list[dict[str, Any]] = []
    invalid: list[dict[str, str]] = []

    for class_name, class_dir in class_dirs.items():
        files = sorted(
            path for path in class_dir.rglob("*")
            if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
        )
        for path in files:
            try:
                with Image.open(path) as image:
                    width, height = image.size
                    mode = image.mode
                    image.convert("RGB").verify()
                records.append(
                    {
                        "split": split_name,
                        "class_name": class_name,
                        "display_name": DISPLAY_NAMES[class_name],
                        "path": str(path),
                        "filename": path.name,
                        "width": width,
                        "height": height,
                        "mode": mode,
                        "extension": path.suffix.lower(),
                    }
                )
            except (UnidentifiedImageError, OSError, ValueError) as exc:
                invalid.append(
                    {
                        "split": split_name,
                        "class_name": class_name,
                        "path": str(path),
                        "error": str(exc),
                    }
                )

    return pd.DataFrame(records), pd.DataFrame(invalid)

train_inventory, train_invalid = scan_dataset(train_class_dirs, "train")
test_inventory, test_invalid = scan_dataset(test_class_dirs, "test")
inventory = pd.concat([train_inventory, test_inventory], ignore_index=True)
invalid_files = pd.concat([train_invalid, test_invalid], ignore_index=True)

print(f"Valid training images: {len(train_inventory):,}")
print(f"Valid testing images:  {len(test_inventory):,}")
print(f"Total valid images:    {len(inventory):,}")
print(f"Invalid images:        {len(invalid_files):,}")

if not invalid_files.empty:
    display(invalid_files)
    invalid_files.to_csv(ARTIFACT_DIR / "invalid_images.csv", index=False)
    raise RuntimeError(
        "Invalid or corrupt images were found. Remove or replace them before training."
    )

if inventory.empty:
    raise RuntimeError("No supported image files were found.")

In [ ]:
distribution = (
    inventory.groupby(["split", "class_name"])
    .size()
    .rename("image_count")
    .reset_index()
)
distribution["display_name"] = distribution["class_name"].map(DISPLAY_NAMES)
distribution = distribution[
    ["split", "class_name", "display_name", "image_count"]
].sort_values(["split", "class_name"])

display(distribution)
distribution.to_csv(ARTIFACT_DIR / "dataset_distribution.csv", index=False)

dimension_summary = inventory[["width", "height"]].describe().round(2)
display(dimension_summary)
dimension_summary.to_csv(ARTIFACT_DIR / "image_dimension_summary.csv")

In [ ]:
pivot_distribution = distribution.pivot(
    index="display_name",
    columns="split",
    values="image_count",
).fillna(0)

ax = pivot_distribution.plot(
    kind="bar",
    figsize=(10, 5),
    title="Brain MRI Dataset Distribution",
)
ax.set_xlabel("Class")
ax.set_ylabel("Number of images")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "dataset_distribution.png", dpi=180, bbox_inches="tight")
plt.show()

## 5. TensorFlow Input Pipelines

The training directory is split deterministically into training and validation subsets. The original `Testing/` directory remains untouched and is used only for final evaluation.

EfficientNet implementations in modern Keras include input rescaling internally, so image tensors remain in the standard `[0, 255]` range.

In [ ]:
IMAGE_SIZE = tuple(CONFIG["training"]["image_size"])
BATCH_SIZE = int(CONFIG["training"]["batch_size"])
VALIDATION_SPLIT = float(CONFIG["training"]["validation_split"])
AUTOTUNE = tf.data.AUTOTUNE

# image_dataset_from_directory sorts directory names alphabetically.
# This explicit mapping is saved and reused by the application.
train_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="categorical",
    validation_split=VALIDATION_SPLIT,
    subset="training",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="categorical",
    validation_split=VALIDATION_SPLIT,
    subset="validation",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_ds = keras.utils.image_dataset_from_directory(
    TEST_DIR,
    labels="inferred",
    label_mode="categorical",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

CLASS_NAMES = list(train_ds.class_names)

if CLASS_NAMES != list(val_ds.class_names) or CLASS_NAMES != list(test_ds.class_names):
    raise ValueError(
        "Class ordering differs across train, validation, and test datasets."
    )

print("TensorFlow class order:", CLASS_NAMES)
print("Display class order:", [DISPLAY_NAMES.get(name, name.title()) for name in CLASS_NAMES])

labels_payload = {
    "class_names": CLASS_NAMES,
    "display_names": [DISPLAY_NAMES.get(name, name.title()) for name in CLASS_NAMES],
    "class_to_index": {name: index for index, name in enumerate(CLASS_NAMES)},
    "index_to_class": {str(index): name for index, name in enumerate(CLASS_NAMES)},
    "model_version": CONFIG["project"]["model_version"],
    "image_size": list(IMAGE_SIZE),
}

labels_path = PROJECT_ROOT / CONFIG["outputs"]["labels"]
labels_path.parent.mkdir(parents=True, exist_ok=True)
labels_path.write_text(json.dumps(labels_payload, indent=2), encoding="utf-8")
print("Saved labels:", labels_path)

In [ ]:
def configure_dataset(dataset: tf.data.Dataset, training: bool = False) -> tf.data.Dataset:
    options = tf.data.Options()
    options.experimental_deterministic = not training
    dataset = dataset.with_options(options)
    return dataset.prefetch(AUTOTUNE)

train_ds = configure_dataset(train_ds, training=True)
val_ds = configure_dataset(val_ds)
test_ds = configure_dataset(test_ds)

for images, labels in train_ds.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    print("Image dtype:", images.dtype)
    print("Label dtype:", labels.dtype)
    print("Image range:", float(tf.reduce_min(images)), "to", float(tf.reduce_max(images)))

## 6. Data Augmentation

The augmentation policy applies conservative geometric and intensity transformations. Aggressive transformations are avoided because MRI anatomy must remain plausible.

In [ ]:
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal", seed=SEED),
        layers.RandomRotation(0.05, fill_mode="reflect", seed=SEED),
        layers.RandomZoom(
            height_factor=(-0.08, 0.08),
            width_factor=(-0.08, 0.08),
            fill_mode="reflect",
            seed=SEED,
        ),
        layers.RandomTranslation(
            height_factor=0.04,
            width_factor=0.04,
            fill_mode="reflect",
            seed=SEED,
        ),
        layers.RandomContrast(0.10, seed=SEED),
    ],
    name="mri_augmentation",
)

In [ ]:
for image_batch, label_batch in train_ds.take(1):
    sample_images = image_batch[:8]
    augmented_images = data_augmentation(sample_images, training=True)

plt.figure(figsize=(14, 7))
for index in range(8):
    plt.subplot(2, 4, index + 1)
    plt.imshow(tf.cast(tf.clip_by_value(augmented_images[index], 0, 255), tf.uint8))
    plt.axis("off")
plt.suptitle("Sample Augmented Training Images", y=1.02)
plt.tight_layout()
plt.show()

## 7. Build EfficientNetB0

The classifier uses:

- EfficientNetB0 pretrained on ImageNet
- Global average pooling
- Batch normalization
- Dense layer with Swish activation
- Dropout
- Four-class softmax output

The softmax layer explicitly uses `float32` so probabilities remain numerically stable under mixed precision.

In [ ]:
NUM_CLASSES = len(CLASS_NAMES)
DENSE_UNITS = int(CONFIG["training"]["dense_units"])
DROPOUT_RATE = float(CONFIG["training"]["dropout"])
LABEL_SMOOTHING = float(CONFIG["training"]["label_smoothing"])

def build_model() -> tuple[keras.Model, keras.Model]:
    inputs = keras.Input(shape=(*IMAGE_SIZE, 3), name="mri_image")
    x = data_augmentation(inputs)

    backbone = keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_tensor=x,
    )
    backbone.trainable = False

    x = backbone.output
    x = layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
    x = layers.BatchNormalization(name="head_batch_norm")(x)
    x = layers.Dense(
        DENSE_UNITS,
        activation="swish",
        kernel_regularizer=keras.regularizers.l2(1e-5),
        name="head_dense",
    )(x)
    x = layers.Dropout(DROPOUT_RATE, seed=SEED, name="head_dropout")(x)
    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax",
        dtype="float32",
        name="predictions",
    )(x)

    model = keras.Model(inputs, outputs, name="brainvision_efficientnetb0")
    return model, backbone

model, backbone = build_model()
model.summary()

In [ ]:
def compile_model(model: keras.Model, learning_rate: float) -> None:
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=keras.losses.CategoricalCrossentropy(
            label_smoothing=LABEL_SMOOTHING
        ),
        metrics=[
            keras.metrics.CategoricalAccuracy(name="accuracy"),
            keras.metrics.TopKCategoricalAccuracy(k=2, name="top_2_accuracy"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
        ],
    )

HEAD_LEARNING_RATE = float(CONFIG["training"]["head_learning_rate"])
compile_model(model, HEAD_LEARNING_RATE)

## 8. Training Callbacks

The best validation-loss checkpoint is saved throughout both training stages. TensorBoard logs and CSV history are also retained.

In [ ]:
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
RUN_LOG_DIR = LOG_DIR / f"efficientnetb0_{RUN_ID}"
RUN_LOG_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = PROJECT_ROOT / CONFIG["outputs"]["best_model"]
FINAL_MODEL_PATH = PROJECT_ROOT / CONFIG["outputs"]["final_model"]
TRAINING_LOG_PATH = ARTIFACT_DIR / f"epoch_log_{RUN_ID}.csv"

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=BEST_MODEL_PATH,
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=False,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=5,
        min_delta=1e-4,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.25,
        patience=2,
        min_lr=1e-7,
        verbose=1,
    ),
    keras.callbacks.TensorBoard(
        log_dir=RUN_LOG_DIR,
        histogram_freq=1,
        write_graph=True,
    ),
    keras.callbacks.CSVLogger(
        filename=TRAINING_LOG_PATH,
        append=True,
    ),
    keras.callbacks.TerminateOnNaN(),
]

print("Best model path:", BEST_MODEL_PATH)
print("TensorBoard log directory:", RUN_LOG_DIR)

## 9. Stage 1 — Train the Classification Head

In [ ]:
HEAD_EPOCHS = int(CONFIG["training"]["head_epochs"])

stage1_start = time.perf_counter()
history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=HEAD_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)
stage1_seconds = time.perf_counter() - stage1_start
print(f"Stage 1 duration: {stage1_seconds / 60:.2f} minutes")

## 10. Stage 2 — Fine-Tune the Upper Backbone

Batch Normalization layers remain frozen during fine-tuning. This prevents their running statistics from becoming unstable on a relatively small medical-image dataset.

In [ ]:
FINE_TUNE_FROM_LAYER = int(CONFIG["training"]["fine_tune_from_layer"])
FINE_TUNE_LEARNING_RATE = float(CONFIG["training"]["fine_tune_learning_rate"])
FINE_TUNE_EPOCHS = int(CONFIG["training"]["fine_tune_epochs"])

backbone.trainable = True

for index, layer in enumerate(backbone.layers):
    should_train = index >= FINE_TUNE_FROM_LAYER
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
    else:
        layer.trainable = should_train

trainable_backbone_layers = sum(layer.trainable for layer in backbone.layers)
print(f"Trainable backbone layers: {trainable_backbone_layers}/{len(backbone.layers)}")

compile_model(model, FINE_TUNE_LEARNING_RATE)

In [ ]:
initial_epoch = len(history_head.epoch)
total_epochs = initial_epoch + FINE_TUNE_EPOCHS

stage2_start = time.perf_counter()
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    initial_epoch=initial_epoch,
    epochs=total_epochs,
    callbacks=callbacks,
    verbose=1,
)
stage2_seconds = time.perf_counter() - stage2_start
print(f"Stage 2 duration: {stage2_seconds / 60:.2f} minutes")

## 11. Consolidate Training History

In [ ]:
def merge_histories(*histories: keras.callbacks.History) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    global_epoch = 0
    for stage_number, history in enumerate(histories, start=1):
        for local_index in range(len(history.epoch)):
            row: dict[str, Any] = {
                "epoch": global_epoch + 1,
                "stage": stage_number,
            }
            for metric_name, values in history.history.items():
                row[metric_name] = float(values[local_index])
            rows.append(row)
            global_epoch += 1
    return pd.DataFrame(rows)

history_df = merge_histories(history_head, history_fine)
history_output_path = PROJECT_ROOT / CONFIG["outputs"]["history"]
history_df.to_csv(history_output_path, index=False)
display(history_df.tail())
print("Saved training history:", history_output_path)

In [ ]:
def plot_training_history(history: pd.DataFrame) -> None:
    metric_pairs = [
        ("accuracy", "val_accuracy", "Accuracy"),
        ("loss", "val_loss", "Loss"),
        ("precision", "val_precision", "Precision"),
        ("recall", "val_recall", "Recall"),
    ]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    for axis, (train_metric, val_metric, title) in zip(axes.flatten(), metric_pairs):
        if train_metric in history.columns:
            axis.plot(history["epoch"], history[train_metric], label=f"Training {title}")
        if val_metric in history.columns:
            axis.plot(history["epoch"], history[val_metric], label=f"Validation {title}")
        axis.axvline(
            x=len(history_head.epoch) + 0.5,
            linestyle="--",
            linewidth=1,
            label="Fine-tuning begins" if title == "Accuracy" else None,
        )
        axis.set_title(title)
        axis.set_xlabel("Epoch")
        axis.grid(alpha=0.25)
        axis.legend()

    fig.suptitle("BrainVision AI Training History", fontsize=16)
    plt.tight_layout()
    output_path = ARTIFACT_DIR / "training_curves.png"
    plt.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.show()

plot_training_history(history_df)

## 12. Restore and Save the Best Model

The best checkpoint is reloaded before final evaluation. A second final-model export is also created for traceability.

In [ ]:
if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(f"Best-model checkpoint was not created: {BEST_MODEL_PATH}")

best_model = keras.models.load_model(BEST_MODEL_PATH)
best_model.save(FINAL_MODEL_PATH)

print("Loaded best model:", BEST_MODEL_PATH)
print("Saved final model:", FINAL_MODEL_PATH)

## 13. Final Test Evaluation

In [ ]:
evaluation_values = best_model.evaluate(test_ds, verbose=1, return_dict=True)
test_metrics = {name: float(value) for name, value in evaluation_values.items()}
display(pd.DataFrame([test_metrics]).round(5))

metrics_output_path = PROJECT_ROOT / CONFIG["outputs"]["metrics"]
metrics_output_path.write_text(
    json.dumps(test_metrics, indent=2),
    encoding="utf-8",
)
print("Saved test metrics:", metrics_output_path)

In [ ]:
y_true_batches: list[np.ndarray] = []
y_probability_batches: list[np.ndarray] = []

for image_batch, label_batch in test_ds:
    probabilities = best_model.predict_on_batch(image_batch)
    y_probability_batches.append(np.asarray(probabilities))
    y_true_batches.append(np.asarray(label_batch))

y_true_one_hot = np.concatenate(y_true_batches, axis=0)
y_probabilities = np.concatenate(y_probability_batches, axis=0)
y_true = np.argmax(y_true_one_hot, axis=1)
y_pred = np.argmax(y_probabilities, axis=1)
y_confidence = np.max(y_probabilities, axis=1)

print("Predictions:", len(y_pred))
print("Probability matrix shape:", y_probabilities.shape)

## 14. Classification Report and Confusion Matrix

In [ ]:
report_dict = classification_report(
    y_true,
    y_pred,
    target_names=[DISPLAY_NAMES.get(name, name.title()) for name in CLASS_NAMES],
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report_dict).transpose()
classification_report_path = PROJECT_ROOT / CONFIG["outputs"]["classification_report"]
report_df.to_csv(classification_report_path)
display(report_df.round(4))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=[DISPLAY_NAMES.get(name, name.title()) for name in CLASS_NAMES],
    columns=[DISPLAY_NAMES.get(name, name.title()) for name in CLASS_NAMES],
)
confusion_matrix_path = PROJECT_ROOT / CONFIG["outputs"]["confusion_matrix"]
cm_df.to_csv(confusion_matrix_path)
display(cm_df)

fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(cm, interpolation="nearest", cmap="Blues")
fig.colorbar(image, ax=ax)
tick_labels = [DISPLAY_NAMES.get(name, name.title()) for name in CLASS_NAMES]
ax.set(
    xticks=np.arange(len(tick_labels)),
    yticks=np.arange(len(tick_labels)),
    xticklabels=tick_labels,
    yticklabels=tick_labels,
    ylabel="True class",
    xlabel="Predicted class",
    title="BrainVision AI Confusion Matrix",
)
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

threshold = cm.max() / 2 if cm.size else 0
for row in range(cm.shape[0]):
    for column in range(cm.shape[1]):
        ax.text(
            column,
            row,
            f"{cm[row, column]:d}",
            ha="center",
            va="center",
            color="white" if cm[row, column] > threshold else "black",
        )

plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "confusion_matrix.png", dpi=180, bbox_inches="tight")
plt.show()

## 15. ROC Curves

In [ ]:
y_true_binary = label_binarize(y_true, classes=np.arange(NUM_CLASSES))

roc_rows: list[dict[str, float | str]] = []
plt.figure(figsize=(9, 7))

for class_index, class_name in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(y_true_binary[:, class_index], y_probabilities[:, class_index])
    class_auc = auc(fpr, tpr)
    display_name = DISPLAY_NAMES.get(class_name, class_name.title())
    plt.plot(fpr, tpr, linewidth=2, label=f"{display_name} (AUC={class_auc:.4f})")
    roc_rows.append({"class_name": class_name, "roc_auc": float(class_auc)})

micro_fpr, micro_tpr, _ = roc_curve(y_true_binary.ravel(), y_probabilities.ravel())
micro_auc = auc(micro_fpr, micro_tpr)
plt.plot(
    micro_fpr,
    micro_tpr,
    linestyle="--",
    linewidth=2,
    label=f"Micro-average (AUC={micro_auc:.4f})",
)

plt.plot([0, 1], [0, 1], linestyle=":", linewidth=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("One-vs-Rest ROC Curves")
plt.legend(loc="lower right")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "roc_curves.png", dpi=180, bbox_inches="tight")
plt.show()

roc_df = pd.DataFrame(roc_rows)
roc_df.loc[len(roc_df)] = {"class_name": "micro_average", "roc_auc": float(micro_auc)}
roc_df.to_csv(ARTIFACT_DIR / "roc_auc_scores.csv", index=False)
display(roc_df)

## 16. Precision-Recall Curves

In [ ]:
pr_rows: list[dict[str, float | str]] = []
plt.figure(figsize=(9, 7))

for class_index, class_name in enumerate(CLASS_NAMES):
    precision_values, recall_values, _ = precision_recall_curve(
        y_true_binary[:, class_index],
        y_probabilities[:, class_index],
    )
    pr_auc = auc(recall_values, precision_values)
    display_name = DISPLAY_NAMES.get(class_name, class_name.title())
    plt.plot(
        recall_values,
        precision_values,
        linewidth=2,
        label=f"{display_name} (AUC={pr_auc:.4f})",
    )
    pr_rows.append({"class_name": class_name, "pr_auc": float(pr_auc)})

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("One-vs-Rest Precision-Recall Curves")
plt.legend(loc="lower left")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "precision_recall_curves.png", dpi=180, bbox_inches="tight")
plt.show()

pr_df = pd.DataFrame(pr_rows)
pr_df.to_csv(ARTIFACT_DIR / "precision_recall_auc_scores.csv", index=False)
display(pr_df)

## 17. Prediction Table and Misclassification Analysis

In [ ]:
test_file_paths = [
    str(path)
    for class_name in test_ds.class_names
    for path in sorted(
        [
            candidate
            for candidate in (test_class_dirs.get(class_name, TEST_DIR / class_name)).rglob("*")
            if candidate.is_file() and candidate.suffix.lower() in SUPPORTED_EXTENSIONS
        ]
    )
]

if len(test_file_paths) != len(y_true):
    print(
        "Warning: Test path count does not match prediction count. "
        "The prediction table will omit file paths."
    )
    test_file_paths = [""] * len(y_true)

prediction_df = pd.DataFrame(
    {
        "file_path": test_file_paths,
        "true_index": y_true,
        "predicted_index": y_pred,
        "true_class": [CLASS_NAMES[index] for index in y_true],
        "predicted_class": [CLASS_NAMES[index] for index in y_pred],
        "confidence": y_confidence,
        "correct": y_true == y_pred,
    }
)

for class_index, class_name in enumerate(CLASS_NAMES):
    prediction_df[f"probability_{class_name}"] = y_probabilities[:, class_index]

prediction_df.to_csv(ARTIFACT_DIR / "predictions.csv", index=False)
display(prediction_df.head())
print("Misclassified images:", int((~prediction_df["correct"]).sum()))

In [ ]:
misclassified = prediction_df.loc[~prediction_df["correct"]].copy()
misclassified = misclassified.sort_values("confidence", ascending=False)

def show_misclassified_examples(frame: pd.DataFrame, maximum: int = 16) -> None:
    if frame.empty:
        print("No test images were misclassified.")
        return

    examples = frame.head(maximum)
    columns = 4
    rows = int(np.ceil(len(examples) / columns))
    plt.figure(figsize=(16, 4 * rows))

    for plot_index, (_, row) in enumerate(examples.iterrows(), start=1):
        plt.subplot(rows, columns, plot_index)
        path = row["file_path"]
        if path and Path(path).exists():
            with Image.open(path) as image:
                plt.imshow(image.convert("RGB"))
        else:
            plt.text(0.5, 0.5, "Image path unavailable", ha="center", va="center")
        plt.title(
            f"True: {DISPLAY_NAMES.get(row['true_class'], row['true_class'])}\n"
            f"Pred: {DISPLAY_NAMES.get(row['predicted_class'], row['predicted_class'])}\n"
            f"Confidence: {row['confidence']:.1%}",
            fontsize=10,
        )
        plt.axis("off")

    plt.suptitle("Highest-Confidence Misclassifications", fontsize=16)
    plt.tight_layout()
    plt.savefig(
        ARTIFACT_DIR / "misclassified_examples.png",
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()

show_misclassified_examples(misclassified)

## 18. Save Run Metadata

The metadata file records the training configuration, environment, dataset size, model paths, durations, and final test metrics. This supports reproducibility and later display in the Streamlit application.

In [ ]:
run_metadata = {
    "project_name": CONFIG["project"]["name"],
    "repository": CONFIG["project"]["repository"],
    "model_version": CONFIG["project"]["model_version"],
    "run_id": RUN_ID,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_root": str(DATASET_ROOT),
    "train_image_count_before_validation_split": int(len(train_inventory)),
    "test_image_count": int(len(test_inventory)),
    "class_names": CLASS_NAMES,
    "display_names": [DISPLAY_NAMES.get(name, name.title()) for name in CLASS_NAMES],
    "image_size": list(IMAGE_SIZE),
    "batch_size": BATCH_SIZE,
    "validation_split": VALIDATION_SPLIT,
    "head_epochs_completed": int(len(history_head.epoch)),
    "fine_tune_epochs_completed": int(len(history_fine.epoch)),
    "head_training_minutes": stage1_seconds / 60,
    "fine_tuning_minutes": stage2_seconds / 60,
    "tensorflow_version": tf.__version__,
    "keras_version": keras.__version__,
    "python_version": sys.version.split()[0],
    "mixed_precision_policy": str(keras.mixed_precision.global_policy()),
    "gpu_devices": [device.name for device in gpus],
    "best_model_path": str(BEST_MODEL_PATH),
    "final_model_path": str(FINAL_MODEL_PATH),
    "test_metrics": test_metrics,
    "roc_auc": roc_df.to_dict(orient="records"),
    "precision_recall_auc": pr_df.to_dict(orient="records"),
}

metadata_output_path = PROJECT_ROOT / CONFIG["outputs"]["run_metadata"]
metadata_output_path.write_text(
    json.dumps(run_metadata, indent=2),
    encoding="utf-8",
)

print("Saved run metadata:", metadata_output_path)

## 19. Final Verification

In [ ]:
required_outputs = [
    BEST_MODEL_PATH,
    FINAL_MODEL_PATH,
    labels_path,
    history_output_path,
    metrics_output_path,
    classification_report_path,
    confusion_matrix_path,
    metadata_output_path,
    ARTIFACT_DIR / "predictions.csv",
    ARTIFACT_DIR / "training_curves.png",
    ARTIFACT_DIR / "confusion_matrix.png",
    ARTIFACT_DIR / "roc_curves.png",
    ARTIFACT_DIR / "precision_recall_curves.png",
]

verification = pd.DataFrame(
    {
        "artifact": [str(path.relative_to(PROJECT_ROOT)) for path in required_outputs],
        "exists": [path.exists() for path in required_outputs],
        "size_bytes": [path.stat().st_size if path.exists() else 0 for path in required_outputs],
    }
)
display(verification)

missing = verification.loc[~verification["exists"], "artifact"].tolist()
if missing:
    raise RuntimeError(f"Training completed, but required outputs are missing: {missing}")

print("BrainVision AI training pipeline completed successfully.")
print(f"Best model: {BEST_MODEL_PATH}")
print(f"Test accuracy: {test_metrics.get('accuracy', float('nan')):.4%}")

## 20. Next Build

Build 2 will add:

- a dedicated Grad-CAM development notebook,
- production inference and preprocessing modules,
- Grad-CAM utility functions,
- model-export validation,
- and application-ready artifact handling.